# Fast Master Sanity Check: All 4 Experiments (E2, E3, E4, E5)
Ultra-fast smoke test (2 rounds each, ~2-3 minutes total on GPU) to verify code correctness, dataset loading, topology engines, Ditto personalization, Hierarchical Ensemble, and Byzantine Defense mechanisms before running full sweeps.

In [ ]:
!git clone https://github.com/nam200718/Topology-aware-FDL.git project_repo
%cd project_repo
!pip install -r requirements.txt
!pip install networkx pytest

In [ ]:
# Run full unit test suite (79 tests)
!pytest tests/ -v

## 1. Fast Sanity E2 & E3: Clean Convergence (2 rounds, Key Topologies)

In [ ]:
import yaml, os

# Sanity E2 (Random Partition)
with open('temp/exp2_random_with_HE.yaml', 'r') as f:
    cfg_e2 = yaml.safe_load(f)
cfg_e2['num_rounds'] = 2
cfg_e2['clients']['local_steps'] = 1
cfg_e2['topologies'] = [
    {'label': 'Star (FedAvg)', 'params': {'personalization_method': 'none'}, 'type': 'star'},
    {'label': 'Star (Ditto)', 'params': {'personalization_method': 'ditto', 'ditto_lambda': 1.0}, 'type': 'star'},
    {'label': 'HE Ensemble', 'params': {'cluster_method': 'update_similarity', 'num_clusters': 5, 'warmup_max_rounds': 10, 'warmup_min_rounds': 2}, 'type': 'hierarchical_ensemble'}
]
cfg_e2['env']['output_dir'] = './outputs/sanity_E2'
with open('temp/sanity_e2.yaml', 'w') as f:
    yaml.dump(cfg_e2, f)

!python temp/run_baseline_comparison.py --config temp/sanity_e2.yaml

## 2. Fast Sanity E4 & E5: Byzantine Defense (2 rounds, q=0.2, Key Topologies)

In [ ]:
# Sanity E5 (Hierarchical Partition + Defense)
with open('temp/exp5_defense_hier.yaml', 'r') as f:
    cfg_e5 = yaml.safe_load(f)
cfg_e5['num_rounds'] = 2
cfg_e5['clients']['local_steps'] = 1
cfg_e5['byzantine_rates'] = [0.2]
cfg_e5['byzantine_types'] = ['label_flip']
cfg_e5['topologies'] = [
    {'label': 'Star (No Defense)', 'params': {'defense_mode': 'none'}, 'type': 'star'},
    {'label': 'Star (Ditto, No Defense)', 'params': {'defense_mode': 'none', 'personalization_method': 'ditto', 'ditto_lambda': 1.0}, 'type': 'star'},
    {'label': 'Ring (Full Defense)', 'params': {'defense_mode': 'soft_cosine'}, 'type': 'ring'},
    {'label': 'HE (Full Defense)', 'params': {'cluster_method': 'update_similarity', 'defense_mode': 'soft_cosine', 'defense_scope': 'both', 'num_clusters': 5, 'warmup_max_rounds': 10, 'warmup_min_rounds': 2}, 'type': 'hierarchical_ensemble'}
]
cfg_e5['env']['output_dir'] = './outputs/sanity_E5'
with open('temp/sanity_e5.yaml', 'w') as f:
    yaml.dump(cfg_e5, f)

!python temp/run_baseline_comparison.py --config temp/sanity_e5.yaml

## 3. Verify Generated Summary CSVs

In [ ]:
import pandas as pd
from IPython.display import display

for exp_name, path in [
    ('Clean Convergence (E2/E3)', './outputs/sanity_E2/comparison_summary.csv'),
    ('Hierarchical Defense (E4/E5)', './outputs/sanity_E5/comparison_summary.csv'),
]:
    print(f'=== Sanity Summary: {exp_name} ===')
    if os.path.exists(path):
        display(pd.read_csv(path))
    else:
        print('File not found:', path)